In [9]:
import yfinance as yf
import polars as pl
import altair
from pathlib import Path

In [10]:
TICKERS_TO_DWNLD = ["NVDA", 
"PYPL", "VZ", "IBM", "2222.SR", "ETH-USD", "LLOY.L", "SHEL.L", "UKOG.L", 
"CHD", "TSN", "INTC", "AMD", "AAPL", "AMZN", "JPM", "ORCL", "PG", "XOM",
"MSFT", "META", "NFLX", "WMT", "LLOY", "HSBA", "BA.", "0700.HK", "1810.HK"
]

INTERVAL = "4h"
PERIOD = "2y"
IND_WINDOW = 14

path = Path(f"stocks/{PERIOD}/{INTERVAL}")
if not path.exists(): path.mkdir(parents=True, exist_ok=True)

In [11]:
def calculate_rsi(close_col: str = "Close") -> pl.Expr:
    delta = pl.col(close_col).diff()
    gain = pl.when(delta > 0).then(delta).otherwise(0.0)
    loss = pl.when(delta < 0).then(-delta).otherwise(0.0)
    avg_gain = gain.ewm_mean(span=IND_WINDOW, adjust=False)
    avg_loss = loss.ewm_mean(span=IND_WINDOW, adjust=False)
    return (100 - (100 / (1 + (avg_gain / avg_loss)))).alias("Rsi")

def stochastic_oscillator(df: pl.DataFrame, smoothing: int = 3) -> pl.DataFrame:
    low_min = pl.col("Low").rolling_min(window_size=IND_WINDOW)
    high_max = pl.col("High").rolling_max(window_size=IND_WINDOW)
    k = ((pl.col("Close") - low_min) / (high_max - low_min)) * 100
    d = k.rolling_mean(window_size=smoothing)
    return df.with_columns(k.alias("%K"), d.alias("%D"))

def plot(ts: pl.DataFrame, name: str):
    # Convert timezone-aware datetimes to UTC for Altair compatibility
    for col in ts.columns:
        dtype = ts.schema[col]
        if isinstance(dtype, pl.Datetime) and getattr(dtype, "time_zone", None):
            ts = ts.with_columns(pl.col(col).dt.convert_time_zone("UTC").alias(col))
    if "Date" in ts.columns and "Datetime" not in ts.columns:
        ts = ts.rename({"Date": "Datetime"})
    data = ts.to_pandas()

    base = altair.Chart(data).encode(
        x=altair.X("Datetime:T", axis=altair.Axis(title="Datetime"))
    )

    close_line = base.mark_line(tooltip=True, color='navy').encode(
        y=altair.Y("Close:Q", axis=altair.Axis(title="Close")),
    )
    price = close_line.properties(
            height=250,
            width=800,
            title=f"{name} {INTERVAL} Close"
    )

    ks = set(ts.columns)
    has_kd = "%K" in ks and "%D" in ks

    if has_kd:
        k_line = base.mark_line(tooltip=True, color='darkorange').encode(
            y=altair.Y("%K:Q")
        )
        d_line = base.mark_line(tooltip=True, color='green').encode(
            y=altair.Y("%D:Q")
        )
        stoch = altair.layer(k_line, d_line).resolve_scale(
            y="independent"
        ).properties(
            height=250,
            width=800,
            title=f"{name} {INTERVAL} %K, %D"
        )

    if "Rsi" in ks:
        rsi = altair.Chart(data).mark_line(tooltip=True, color='purple').encode(
            x="Datetime:T",
            y=altair.Y("Rsi:Q", axis=altair.Axis(title="RSI"))
        ).properties(
            height=100,
            width=800,
            title="RSI"
        )
    else:
        rsi = None

    if "Dividends" in ks:
        dividends = altair.Chart(data).mark_bar(color="gray", opacity=0.7).encode(
            x="Datetime:T",
            y=altair.Y("Dividends:Q", axis=altair.Axis(title="Dividends"))
        ).properties(
            height=60,
            width=800,
            title="Dividends"
        )
    else:
        dividends = None

    charts = [price, stoch]
    if rsi is not None:
        charts.append(rsi)
    if dividends is not None:
        charts.append(dividends)

    full_chart = altair.vconcat(*charts).configure_scale(zero=False).resolve_scale(
        y="independent"
    ).add_params(
        altair.selection_interval(bind='scales', encodings=['x'])
    )

    return full_chart

In [12]:
for name in TICKERS_TO_DWNLD:
    try:
        ticker = yf.Ticker(name)
        dividends = ticker.dividends

        pd_data = ticker.history(period=PERIOD, interval=INTERVAL)
        price_data = pl.from_pandas(pd_data.reset_index())

    except:
        print("Could not install " + name)
        continue

    if "Dividends" not in price_data.columns:
        price_data = price_data.with_columns(pl.lit(0.0).alias("Dividends"))
    price_data = price_data.with_columns(
        (pl.col("Dividends").fill_null(0) != 0.0).cast(pl.Float64).alias("Dividends")
    )
    price_data = price_data.with_columns(calculate_rsi())
    price_data = stochastic_oscillator(price_data)

    # Only drop columns that exist in the dataframe
    columns_to_drop = ["High", "Low", "Open", "Volume", "Stock Splits"]
    existing_columns_to_drop = [col for col in columns_to_drop if col in price_data.columns]
    if existing_columns_to_drop:
        price_data = price_data.drop(existing_columns_to_drop)
    
    # Check if there are enough rows for slicing (need at least IND_WINDOW + 1 rows)
    min_rows_needed = IND_WINDOW + 1
    if len(price_data) <= min_rows_needed:
        print(f"Skipping {name}: insufficient data ({len(price_data)} rows, need at least {min_rows_needed + 1})")
        continue
    
    # Remove the first IND_WINDOW + 1 rows (needed for rolling calculations)
    price_data = price_data.slice(IND_WINDOW + 1, len(price_data) - IND_WINDOW - 1)
    price_data.write_parquet(f"stocks/{PERIOD}/{INTERVAL}/{name}.parquet")
    plot(price_data, name)

$LLOY: possibly delisted; no timezone found
$LLOY: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$HSBA: possibly delisted; no timezone found


Skipping LLOY: insufficient data (0 rows, need at least 16)


$HSBA: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$BA.: possibly delisted; no timezone found


Skipping HSBA: insufficient data (0 rows, need at least 16)


$BA.: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


Skipping BA.: insufficient data (0 rows, need at least 16)


In [13]:
# Data saved as Parquet in loop above (stocks/{PERIOD}/{INTERVAL}/*.parquet)